In [1]:
import networkx as nx
import msgpack

    # Importing Matplotlib, Pandas, and NumPy for logs parsing and visualization
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import sys
import os

In [2]:
root_dir = os.path.dirname(os.path.dirname(os.path.abspath("./test1_data.json")))
sys.path.append(root_dir)

from edge_sim_py import *
from edge_sim_py import flow_scheduling

In [4]:
#Dispaly components
def Collect_Components()->dict:
    datasets={}
    #User
    datasets[f"{User.__name__}"]=[]
    for user in User.all():
        datasets[f"{user.__class__.__name__}"].append((user._to_dict()))
        
    datasets[f"{Application.__name__}"]=[]
    for app in Application.all():
        datasets[f"{app.__class__.__name__}"].append((app._to_dict()))
        
    datasets[f"{Service.__name__}"]=[]
    for service in Service.all():
        datasets[f"{service.__class__.__name__}"].append((service._to_dict()))
        
    datasets[f"{EdgeServer.__name__}"]=[]
    for server in EdgeServer.all():
        datasets[f"{server.__class__.__name__}"].append((server._to_dict()))
       
    datasets[f"{BaseStation.__name__}"]=[]    
    for station in BaseStation.all():
        datasets[f"{station.__class__.__name__}"].append((station._to_dict()))  
        
    datasets[f"{NetworkSwitch.__name__}"]=[]         
    for switch in NetworkSwitch.all():
        datasets[f"{switch.__class__.__name__}"].append((switch._to_dict()))
        
    datasets[f"{NetworkLink.__name__}"]=[]           
    for link in NetworkLink.all():
        datasets[f"{link.__class__.__name__}"].append((link._to_dict()))  

    return datasets
#

def printComponent(datasets,category:str):
    print(f"{category}:")
    for agent in datasets[category]:
        print(agent)
        print()
    
def printClass(category):
    print(category.__name__)
    #User
    for agent in category.all():
        print(agent._to_dict())
        print()
        
    

In [5]:
#2.步进测试
# 先替换各类的步进函数
#User
User.step = tools.User_step
User.set_communication_path = tools.User_path
#app and service
Application.step = tools.Application_Step
Service.step = tools.Service_Step
Service.provision = tools.Service_Provision
#networkflow
NetworkFlow.step = tools.NetworkFlow_Step
#server
EdgeServer.step = tools.EdgeServer_Step
EdgeServer.has_capacity_to_host = tools.has_capacity_to_host
#simulator
Simulator.step = tools.Simulator_Step
#networkswitch
NetworkSwitch.addQueue = tools.addQueue

In [6]:
#1.导入测试
#newworkflow schedule algorithm
simulate = Simulator(
    network_flow_scheduling_algorithm = flow_share
)
simulate.initialize(input_file="./test1_date.json")
datasets = Collect_Components()
printComponent(datasets,"User")
printComponent(datasets,"Application")
printComponent(datasets,"Service")
printComponent(datasets,"EdgeServer")
printComponent(datasets,"BaseStation")
printComponent(datasets,"NetworkSwitch")

User:
{'attributes': {'id': 1, 'coordinates': [0, 3], 'coordinates_trace': [], 'delays': {}, 'delay_slas': {'1': 45}, 'communication_paths': {}, 'making_requests': {}, 'mobility_model_parameters': {}}, 'relationships': {'access_patterns': {'1': {'class': 'CircularDurationAndIntervalAccessPattern', 'id': 1}}, 'mobility_model': None, 'applications': [{'class': 'Application', 'id': 1}], 'base_station': {'class': 'BaseStation', 'id': 1}}}

{'attributes': {'id': 2, 'coordinates': [0, 3], 'coordinates_trace': [], 'delays': {}, 'delay_slas': {'2': 30}, 'communication_paths': {}, 'making_requests': {}, 'mobility_model_parameters': {}}, 'relationships': {'access_patterns': {'2': {'class': 'CircularDurationAndIntervalAccessPattern', 'id': 2}}, 'mobility_model': None, 'applications': [{'class': 'Application', 'id': 2}], 'base_station': {'class': 'BaseStation', 'id': 1}}}

{'attributes': {'id': 3, 'coordinates': [0, 0], 'coordinates_trace': [], 'delays': {}, 'delay_slas': {'3': 30}, 'communication

In [7]:
printClass(NetworkLink)

NetworkLink
{'attributes': {'id': 1, 'delay': 5, 'bandwidth': 10000, 'bandwidth_demand(Mb/s)': 0, 'active': True}, 'relationships': {'active_flows': [], 'applications': [], 'nodes': [{'class': 'NetworkSwitch', 'id': 1}, {'class': 'NetworkSwitch', 'id': 3}]}}

{'attributes': {'id': 2, 'delay': 5, 'bandwidth': 10000, 'bandwidth_demand(Mb/s)': 0, 'active': True}, 'relationships': {'active_flows': [], 'applications': [], 'nodes': [{'class': 'NetworkSwitch', 'id': 2}, {'class': 'NetworkSwitch', 'id': 3}]}}

{'attributes': {'id': 3, 'delay': 5, 'bandwidth': 10000, 'bandwidth_demand(Mb/s)': 0, 'active': True}, 'relationships': {'active_flows': [], 'applications': [], 'nodes': [{'class': 'NetworkSwitch', 'id': 3}, {'class': 'NetworkSwitch', 'id': 4}]}}

{'attributes': {'id': 4, 'delay': 5, 'bandwidth': 10000, 'bandwidth_demand(Mb/s)': 0, 'active': True}, 'relationships': {'active_flows': [], 'applications': [], 'nodes': [{'class': 'NetworkSwitch', 'id': 4}, {'class': 'NetworkSwitch', 'id': 5}]

In [ ]:
# 然后进行仿真模拟
# 首先仿真器步数加1
simulate.schedule.steps+=1
simulate.schedule.time+=1
# 所有用户步进
for usr in User.all():
    usr.step()

printClass(User)

In [ ]:
print(simulate.current_services)

In [ ]:

#假设将服�?1部署在服务器1上（放置于等待队列中）
service1 = Service.all()[0]
server1 = EdgeServer.all()[0]
service1.provision(target_server = server1)  #compute shortest path while provision

In [ ]:
#服务步进
for service in Service.all():
    service.step()
printClass(Service)
print(Service.all()[0]._Service__migrations)

In [ ]:
# 应用步进
for app in Application.all():
    app.step()
printClass(Application)

In [ ]:
# 服务器步进
for server in EdgeServer.all():
    server.step()

printClass(EdgeServer)
#部署服务后服务没有和服务器相关联(在哪一步实现关�??)
'''
1.服务确定部署服务器后就关�?:由provision实现
2.当服务器检测到等待队列中有待传输服务时实现关联：由EdgeServer.step实现
3.当服务对应流量下载完成后才实现关�?:由NetworkFlow.step实现
采用�?1种实现方式
'''

In [ ]:
printClass(Service)

In [ ]:
print(Service.all()[0].path)

NetworkSwitch_1
NetworkSwitch_5
[NetworkSwitch_1, NetworkSwitch_3, NetworkSwitch_4, NetworkSwitch_5]


In [16]:
#网络流步进


In [ ]:
#交换机步�?

In [18]:

#链路步进